# Part 3 — Extension with Temporal User History and Attention Mechanism

This notebook uses **MetadataMLP** directly as the base model, then extends it in two stages:

1. **MetadataMLP** — metadata-aware baseline
2. **SequentialMetadataMLP** — adds temporal user history with a GRU encoder
3. **AttentionSequentialMetadataMLP** — replaces the fixed GRU summary with target-aware attention over user history

All three models are trained and compared in the same pipeline.


In [ ]:
import copy
import math
import os
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
os.makedirs('./checkpoints', exist_ok=True)


Using device: cuda


In [ ]:
METADATA_CATEGORICAL_COLS = ['beer/style', 'beer/brewerId']
METADATA_NUMERIC_COLS = ['beer/ABV']
USER_COL = 'review/profileName'
ITEM_COL = 'beer/beerId'
RATING_COL = 'review/overall'

RAW_TRAIN = pd.read_json('./data/beeradvocate_train.json', lines=True)
RAW_VAL = pd.read_json('./data/beeradvocate_val.json', lines=True)
RAW_TEST = pd.read_json('./data/beeradvocate_test.json', lines=True)

required_cols = [USER_COL, ITEM_COL, RATING_COL] + METADATA_CATEGORICAL_COLS + METADATA_NUMERIC_COLS
missing = [c for c in required_cols if c not in RAW_TRAIN.columns]
if missing:
    raise ValueError(f'Missing required columns in training data: {missing}')

def prepare_split(df):
    split = df[required_cols].copy()
    split = split.dropna(subset=[USER_COL, ITEM_COL, RATING_COL])
    for col in METADATA_CATEGORICAL_COLS:
        split[col] = split[col].fillna('<UNK>').astype(str)
    for col in METADATA_NUMERIC_COLS:
        split[col] = pd.to_numeric(split[col], errors='coerce')
    return split

train_df = prepare_split(RAW_TRAIN)
val_df = prepare_split(RAW_VAL)
test_df = prepare_split(RAW_TEST)

for col in METADATA_NUMERIC_COLS:
    median_value = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_value)
    val_df[col] = val_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

user2idx = {u: i for i, u in enumerate(train_df[USER_COL].unique())}
item2idx = {it: i for i, it in enumerate(train_df[ITEM_COL].unique())}
unknown_user_idx = len(user2idx)
unknown_item_idx = len(item2idx)
num_users = len(user2idx) + 1
num_items = len(item2idx) + 1

categorical_maps = {}
categorical_sizes = {}
for col in METADATA_CATEGORICAL_COLS:
    values = sorted(train_df[col].unique().tolist())
    mapping = {v: i for i, v in enumerate(values)}
    categorical_maps[col] = mapping
    categorical_sizes[col] = len(mapping) + 1

numeric_stats = {}
for col in METADATA_NUMERIC_COLS:
    mean = float(train_df[col].mean())
    std = float(train_df[col].std())
    if std < 1e-8:
        std = 1.0
    numeric_stats[col] = (mean, std)


def encode_split(df):
    out = pd.DataFrame()
    out['user_idx'] = df[USER_COL].map(lambda x: user2idx.get(x, unknown_user_idx)).astype(np.int64)
    out['item_idx'] = df[ITEM_COL].map(lambda x: item2idx.get(x, unknown_item_idx)).astype(np.int64)
    out['rating'] = df[RATING_COL].astype(np.float32)

    for col in METADATA_CATEGORICAL_COLS:
        mapping = categorical_maps[col]
        out[f'{col}__idx'] = df[col].map(lambda x: mapping.get(x, len(mapping))).astype(np.int64)

    for col in METADATA_NUMERIC_COLS:
        mean, std = numeric_stats[col]
        out[f'{col}__num'] = ((df[col].astype(np.float32) - mean) / std).astype(np.float32)
    return out

train_encoded = encode_split(train_df)
val_encoded = encode_split(val_df)
test_encoded = encode_split(test_df)

print('Train encoded:', train_encoded.shape)
print('Val encoded:', val_encoded.shape)
print('Test encoded:', test_encoded.shape)
print('Users:', num_users, 'Items:', num_items)
print('Metadata categorical sizes:', categorical_sizes)


TIMESTAMP_CANDIDATES = ['review/timeUnix', 'review/time', 'review/dateCreated', 'review/date']
timestamp_col = next((col for col in TIMESTAMP_CANDIDATES if col in RAW_TRAIN.columns), None)
if timestamp_col is None:
    raise ValueError(f'No timestamp column found. Checked: {TIMESTAMP_CANDIDATES}')
print('Using timestamp column:', timestamp_col)

required_with_time = required_cols + [timestamp_col]

def prepare_temporal_split(df):
    split = df[required_with_time].copy()
    split = split.dropna(subset=[USER_COL, ITEM_COL, RATING_COL, timestamp_col])
    for col in METADATA_CATEGORICAL_COLS:
        split[col] = split[col].fillna('<UNK>').astype(str)
    for col in METADATA_NUMERIC_COLS:
        split[col] = pd.to_numeric(split[col], errors='coerce')
    split[timestamp_col] = pd.to_numeric(split[timestamp_col], errors='coerce')
    split = split.dropna(subset=[timestamp_col])
    return split

train_time_df = prepare_temporal_split(RAW_TRAIN)
val_time_df = prepare_temporal_split(RAW_VAL)
test_time_df = prepare_temporal_split(RAW_TEST)
for col in METADATA_NUMERIC_COLS:
    median_value = train_time_df[col].median()
    train_time_df[col] = train_time_df[col].fillna(median_value)
    val_time_df[col] = val_time_df[col].fillna(median_value)
    test_time_df[col] = test_time_df[col].fillna(median_value)

def encode_temporal_split(df):
    out = encode_split(df)
    out['timestamp'] = df[timestamp_col].astype(np.int64).values
    return out

train_time_encoded = encode_temporal_split(train_time_df)
val_time_encoded = encode_temporal_split(val_time_df)
test_time_encoded = encode_temporal_split(test_time_df)


Train encoded: (1269292, 6)
Val encoded: (158661, 6)
Test encoded: (158661, 6)
Users: 27573 Items: 54912
Metadata categorical sizes: {'beer/style': 105, 'beer/brewerId': 5232}
Using timestamp column: review/time


In [ ]:
class MetadataRatingsDataset(Dataset):
    def __init__(self, df, categorical_cols, numeric_cols):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
        self.categorical_cols = list(categorical_cols)
        self.numeric_cols = list(numeric_cols)
        self.cat_features = {
            col: torch.tensor(df[f'{col}__idx'].values, dtype=torch.long)
            for col in self.categorical_cols
        }
        self.num_features = {
            col: torch.tensor(df[f'{col}__num'].values, dtype=torch.float32)
            for col in self.numeric_cols
        }

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        cat = {col: values[idx] for col, values in self.cat_features.items()}
        num = {col: values[idx] for col, values in self.num_features.items()}
        return self.users[idx], self.items[idx], self.ratings[idx], cat, num


def metadata_collate_fn(batch):
    users, items, ratings, cat_features, num_features = zip(*batch)
    users = torch.stack(users)
    items = torch.stack(items)
    ratings = torch.stack(ratings)
    cat_batch = {
        col: torch.stack([features[col] for features in cat_features])
        for col in METADATA_CATEGORICAL_COLS
    }
    num_batch = {
        col: torch.stack([features[col] for features in num_features])
        for col in METADATA_NUMERIC_COLS
    }
    return users, items, ratings, cat_batch, num_batch


def make_metadata_loaders(batch_size=256):
    train_dataset = MetadataRatingsDataset(train_encoded, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS)
    val_dataset = MetadataRatingsDataset(val_encoded, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS)
    test_dataset = MetadataRatingsDataset(test_encoded, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=metadata_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=metadata_collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=metadata_collate_fn)
    return train_loader, val_loader, test_loader


def move_feature_dict_to_device(feature_dict, device, kind='long'):
    dtype = torch.long if kind == 'long' else torch.float32
    return {k: v.to(device=device, dtype=dtype) for k, v in feature_dict.items()}


In [ ]:
class MetadataEncoder(nn.Module):
    def __init__(self, embedding_dim, categorical_sizes, numeric_cols):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.categorical_cols = list(categorical_sizes.keys())
        self.numeric_cols = list(numeric_cols)
        self.categorical_embeddings = nn.ModuleDict({
            col: nn.Embedding(size, embedding_dim) for col, size in categorical_sizes.items()
        })
        for emb in self.categorical_embeddings.values():
            nn.init.normal_(emb.weight, std=0.01)
        self.numeric_projection = nn.Linear(len(self.numeric_cols), embedding_dim) if self.numeric_cols else None
        num_pieces = len(self.categorical_cols) + (1 if self.numeric_projection is not None else 0)
        self.output_projection = nn.Linear(embedding_dim * max(1, num_pieces), embedding_dim)

    def forward(self, categorical_features, numeric_features):
        pieces = []
        for col in self.categorical_cols:
            pieces.append(self.categorical_embeddings[col](categorical_features[col]))
        if self.numeric_projection is not None:
            num_tensor = torch.stack([numeric_features[col] for col in self.numeric_cols], dim=1)
            pieces.append(self.numeric_projection(num_tensor))
        return self.output_projection(torch.cat(pieces, dim=1))


class MetadataMLP(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, hidden_dims, dropout, categorical_sizes, numeric_cols):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        self.metadata_encoder = MetadataEncoder(embedding_dim, categorical_sizes, numeric_cols)
        self.mlp_input_dim = embedding_dim * 3
        self.mlp = self._build_mlp(self.mlp_input_dim, hidden_dims, dropout)
        last_dim = hidden_dims[-1] if hidden_dims else self.mlp_input_dim
        self.output_layer = nn.Linear(last_dim, 1)

    def _build_mlp(self, input_dim, hidden_dims, dropout):
        layers = []
        current_dim = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(current_dim, h), nn.ReLU(), nn.Dropout(dropout)])
            current_dim = h
        return nn.Sequential(*layers) if layers else nn.Identity()

    def encode_history(self, item_ids, history_items, history_lengths):
        return None

    def forward(self, user_ids, item_ids, categorical_features, numeric_features, history_items=None, history_lengths=None):
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)
        metadata_vec = self.metadata_encoder(categorical_features, numeric_features)
        pieces = [user_vec, item_vec, metadata_vec]
        history_vec = self.encode_history(item_ids, history_items, history_lengths)
        if history_vec is not None:
            pieces.append(history_vec)
        x = torch.cat(pieces, dim=1)
        return self.output_layer(self.mlp(x)).squeeze(1)


class SequentialMetadataMLP(MetadataMLP):
    def __init__(self, num_users, num_items, embedding_dim, hidden_dims, dropout, categorical_sizes, numeric_cols, sequence_hidden_dim=64):
        super().__init__(num_users, num_items, embedding_dim, hidden_dims, dropout, categorical_sizes, numeric_cols)
        self.history_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.history_embedding.weight, std=0.01)
        self.history_gru = nn.GRU(embedding_dim, sequence_hidden_dim, batch_first=True)
        self.history_projection = nn.Linear(sequence_hidden_dim, embedding_dim)
        self.mlp_input_dim = embedding_dim * 4
        self.mlp = self._build_mlp(self.mlp_input_dim, hidden_dims, dropout)
        last_dim = hidden_dims[-1] if hidden_dims else self.mlp_input_dim
        self.output_layer = nn.Linear(last_dim, 1)

    def encode_history(self, item_ids, history_items, history_lengths):
        if history_items is None or history_lengths is None:
            return None
        original_lengths = history_lengths
        safe_lengths = history_lengths.clamp(min=1)
        embedded = self.history_embedding(history_items)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            safe_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, hidden = self.history_gru(packed)
        history_vec = self.history_projection(hidden[-1])
        empty_mask = (original_lengths == 0).unsqueeze(1)
        history_vec = torch.where(empty_mask, torch.zeros_like(history_vec), history_vec)
        return history_vec


class AttentionSequentialMetadataMLP(SequentialMetadataMLP):
    def __init__(self, num_users, num_items, embedding_dim, hidden_dims, dropout, categorical_sizes, numeric_cols, sequence_hidden_dim=64):
        super().__init__(num_users, num_items, embedding_dim, hidden_dims, dropout, categorical_sizes, numeric_cols, sequence_hidden_dim)
        self.query_projection = nn.Linear(embedding_dim, sequence_hidden_dim)
        self.key_projection = nn.Linear(embedding_dim, sequence_hidden_dim)
        self.value_projection = nn.Linear(embedding_dim, sequence_hidden_dim)
        self.context_projection = nn.Linear(sequence_hidden_dim, embedding_dim)

    def encode_history(self, item_ids, history_items, history_lengths):
        if history_items is None or history_lengths is None:
            return None
        hist_emb = self.history_embedding(history_items)
        query = self.query_projection(self.item_embedding(item_ids)).unsqueeze(1)
        keys = self.key_projection(hist_emb)
        values = self.value_projection(hist_emb)
        scores = torch.matmul(query, keys.transpose(1, 2)).squeeze(1) / math.sqrt(keys.size(-1))
        max_len = history_items.size(1)
        mask = torch.arange(max_len, device=history_lengths.device).unsqueeze(0) >= history_lengths.unsqueeze(1)
        scores = scores.masked_fill(mask, -1e9)
        attention = torch.softmax(scores, dim=1)
        context = torch.bmm(attention.unsqueeze(1), values).squeeze(1)
        context = self.context_projection(context)
        empty_mask = (history_lengths == 0).unsqueeze(1)
        context = torch.where(empty_mask, torch.zeros_like(context), context)
        return context


In [ ]:
SEQUENCE_MAX_LEN = 10


def sort_by_time(df):
    return df.sort_values(['user_idx', 'timestamp', 'item_idx']).reset_index(drop=True)


def attach_histories(source_df, target_df, max_len=10):
    source_df = sort_by_time(source_df)
    target_df = sort_by_time(target_df).copy()
    histories = []
    user_histories = defaultdict(list)
    source_iter = iter(source_df.itertuples(index=False))
    current_row = next(source_iter, None)

    for row in target_df.itertuples(index=False):
        while current_row is not None and (
            int(current_row.user_idx) < int(row.user_idx) or
            (int(current_row.user_idx) == int(row.user_idx) and int(current_row.timestamp) < int(row.timestamp)) or
            (int(current_row.user_idx) == int(row.user_idx) and int(current_row.timestamp) == int(row.timestamp) and int(current_row.item_idx) < int(row.item_idx))
        ):
            user_histories[int(current_row.user_idx)].append(int(current_row.item_idx))
            current_row = next(source_iter, None)
        histories.append(user_histories[int(row.user_idx)][-max_len:])
    target_df['history_items'] = histories
    return target_df


train_seq = attach_histories(train_time_encoded, train_time_encoded, SEQUENCE_MAX_LEN)
val_context = pd.concat([train_time_encoded, val_time_encoded], ignore_index=True)
val_seq = attach_histories(val_context, val_time_encoded, SEQUENCE_MAX_LEN)
test_context = pd.concat([train_time_encoded, val_time_encoded, test_time_encoded], ignore_index=True)
test_seq = attach_histories(test_context, test_time_encoded, SEQUENCE_MAX_LEN)


In [ ]:
class SequentialMetadataDataset(Dataset):
    def __init__(self, df, max_len):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
        self.histories = df['history_items'].tolist()
        self.max_len = max_len
        self.cat_features = {
            col: torch.tensor(df[f'{col}__idx'].values, dtype=torch.long)
            for col in METADATA_CATEGORICAL_COLS
        }
        self.num_features = {
            col: torch.tensor(df[f'{col}__num'].values, dtype=torch.float32)
            for col in METADATA_NUMERIC_COLS
        }

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        hist = list(self.histories[idx])[-self.max_len:]
        hist_len = len(hist)
        padded = hist + [unknown_item_idx] * (self.max_len - hist_len)
        cat = {col: values[idx] for col, values in self.cat_features.items()}
        num = {col: values[idx] for col, values in self.num_features.items()}
        return (
            self.users[idx], self.items[idx], self.ratings[idx], cat, num,
            torch.tensor(padded, dtype=torch.long), torch.tensor(hist_len, dtype=torch.long)
        )


def sequential_collate_fn(batch):
    users, items, ratings, cat_features, num_features, history_items, history_lengths = zip(*batch)
    return (
        torch.stack(users),
        torch.stack(items),
        torch.stack(ratings),
        {col: torch.stack([features[col] for features in cat_features]) for col in METADATA_CATEGORICAL_COLS},
        {col: torch.stack([features[col] for features in num_features]) for col in METADATA_NUMERIC_COLS},
        torch.stack(history_items),
        torch.stack(history_lengths),
    )


def make_sequential_loaders(batch_size=256):
    train_dataset = SequentialMetadataDataset(train_seq, SEQUENCE_MAX_LEN)
    val_dataset = SequentialMetadataDataset(val_seq, SEQUENCE_MAX_LEN)
    test_dataset = SequentialMetadataDataset(test_seq, SEQUENCE_MAX_LEN)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=sequential_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=sequential_collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=sequential_collate_fn)
    return train_loader, val_loader, test_loader


In [ ]:
def evaluate_sequence_rmse(model, data_loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for users, items, ratings, cat_features, num_features, history_items, history_lengths in data_loader:
            users = users.to(device)
            items = items.to(device)
            ratings = ratings.to(device)
            cat_features = move_feature_dict_to_device(cat_features, device, 'long')
            num_features = move_feature_dict_to_device(num_features, device, 'float')
            history_items = history_items.to(device)
            history_lengths = history_lengths.to(device)
            outputs = torch.clamp(model(users, items, cat_features, num_features, history_items, history_lengths), 1.0, 5.0)
            preds.extend(outputs.cpu().numpy())
            targets.extend(ratings.cpu().numpy())
    rmse = math.sqrt(mean_squared_error(targets, preds))
    mae = mean_absolute_error(targets, preds)
    return rmse, mae


def train_sequence_model(model, train_loader, val_loader, epochs=8, lr=1e-3, weight_decay=1e-5):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_state = copy.deepcopy(model.state_dict())
    best_val_rmse = float('inf')
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for users, items, ratings, cat_features, num_features, history_items, history_lengths in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False):
            users = users.to(device)
            items = items.to(device)
            ratings = ratings.to(device)
            cat_features = move_feature_dict_to_device(cat_features, device, 'long')
            num_features = move_feature_dict_to_device(num_features, device, 'float')
            history_items = history_items.to(device)
            history_lengths = history_lengths.to(device)
            optimizer.zero_grad()
            outputs = model(users, items, cat_features, num_features, history_items, history_lengths)
            loss = criterion(outputs, ratings)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * users.size(0)
        avg_loss = total_loss / len(train_loader.dataset)
        val_rmse, val_mae = evaluate_sequence_rmse(model, val_loader)
        print(f'Epoch {epoch+1}/{epochs} - train_loss={avg_loss:.4f} - val_rmse={val_rmse:.4f} - val_mae={val_mae:.4f}')
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return best_val_rmse


CONFIG = {
    'embedding_dim': 64,
    'hidden_dims': [128, 64, 32],
    'dropout': 0.2,
    'lr': 1e-3,
    'weight_decay': 1e-5,
    'batch_size': 256,
    'epochs': 8,
    'sequence_hidden_dim': 64,
}

train_loader, val_loader, test_loader = make_sequential_loaders(CONFIG['batch_size'])


In [ ]:
results = {}

base_model = MetadataMLP(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_dims=CONFIG['hidden_dims'],
    dropout=CONFIG['dropout'],
    categorical_sizes=categorical_sizes,
    numeric_cols=METADATA_NUMERIC_COLS,
).to(device)
train_sequence_model(base_model, train_loader, val_loader, epochs=CONFIG['epochs'], lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
results['MetadataMLP'] = dict(zip(['val_rmse', 'val_mae'], evaluate_sequence_rmse(base_model, val_loader)))
results['MetadataMLP'].update(dict(zip(['test_rmse', 'test_mae'], evaluate_sequence_rmse(base_model, test_loader))))

seq_model = SequentialMetadataMLP(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_dims=CONFIG['hidden_dims'],
    dropout=CONFIG['dropout'],
    categorical_sizes=categorical_sizes,
    numeric_cols=METADATA_NUMERIC_COLS,
    sequence_hidden_dim=CONFIG['sequence_hidden_dim'],
).to(device)
train_sequence_model(seq_model, train_loader, val_loader, epochs=CONFIG['epochs'], lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
results['MetadataMLP_plus_GRUHistory'] = dict(zip(['val_rmse', 'val_mae'], evaluate_sequence_rmse(seq_model, val_loader)))
results['MetadataMLP_plus_GRUHistory'].update(dict(zip(['test_rmse', 'test_mae'], evaluate_sequence_rmse(seq_model, test_loader))))

attn_model = AttentionSequentialMetadataMLP(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_dims=CONFIG['hidden_dims'],
    dropout=CONFIG['dropout'],
    categorical_sizes=categorical_sizes,
    numeric_cols=METADATA_NUMERIC_COLS,
    sequence_hidden_dim=CONFIG['sequence_hidden_dim'],
).to(device)
train_sequence_model(attn_model, train_loader, val_loader, epochs=CONFIG['epochs'], lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
results['MetadataMLP_plus_AttentionHistory'] = dict(zip(['val_rmse', 'val_mae'], evaluate_sequence_rmse(attn_model, val_loader)))
results['MetadataMLP_plus_AttentionHistory'].update(dict(zip(['test_rmse', 'test_mae'], evaluate_sequence_rmse(attn_model, test_loader))))

comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df[['val_rmse', 'val_mae', 'test_rmse', 'test_mae']].sort_values('val_rmse')
comparison_df


Epoch 1/8:   0%|          | 0/4959 [00:00<?, ?it/s]

Epoch 1/8 - train_loss=0.6571 - val_rmse=0.5980 - val_mae=0.4375


Epoch 2/8:   0%|          | 0/4959 [00:00<?, ?it/s]

Epoch 2/8 - train_loss=0.4128 - val_rmse=0.5896 - val_mae=0.4347


Epoch 3/8:   0%|          | 0/4959 [00:00<?, ?it/s]

...

Epoch 7/8:   0%|          | 0/4959 [00:00<?, ?it/s]

Epoch 7/8 - train_loss=0.3410 - val_rmse=0.5935 - val_mae=0.4354


Epoch 8/8:   0%|          | 0/4959 [00:00<?, ?it/s]

Epoch 8/8 - train_loss=0.3356 - val_rmse=0.5913 - val_mae=0.4375


,val_rmse,val_mae,test_rmse,test_mae
MetadataMLP_plus_AttentionHistory,0.586895,0.434979,0.597151,0.460223
MetadataMLP_plus_GRUHistory,0.587682,0.433957,0.592348,0.453791
MetadataMLP,0.589602,0.434673,0.590780,0.449839
